# Simplifying Machine Learning with PyCaret: A Low-code Approach
---


#### <a href="https://hossainlab.github.io">Md. Jubayer Hossain</a>
Founder & Executive Director, CHIRAL Bangladesh 

---

## What is PyCaret? 
- PyCaret is an open-source, low-code machine learning library in Python that automates machine learning workflows 
- PyCaret can be used to replace hundreds of lines of code with few lines only. You spend less time coding and more time on analysis 
- PyCaret is essential a Python wrapper around several machine learning libraries and frameworks such as scikit-learn, XGBoost, CatBoost and few more. 

## Regression with PyCaret - Creating the Whole Machine Learning Pipeline with PyCaret
---
This lecture provides a comprehensive walkthrough of the end-to-end machine learning (ML) pipeline. It covers all essential stages—from data ingestion and preprocessing, through model training and hyperparameter optimization, to prediction and model persistence for future deployment.

The entire workflow is streamlined into fewer than ten intuitive and semantically expressive commands, enabling efficient execution and ease of reproducibility. These commands include:

- `create_model()`: Instantiates and trains a baseline model.
- `tune_model()`: Optimizes model performance through hyperparameter tuning.
- `compare_models()`: Benchmarks multiple models to identify the most performant one.
- `plot_model()`: Generates diagnostic plots for model interpretation.
- `evaluate_model()`: Assesses model performance using various metrics.
- `predict_model()`: Applies the trained model to new, unseen data.

The following sections will walk through each of these steps, offering a complete picture of a streamlined and reproducible machine learning workflow.



<center><img src="../workflow/pycaret-workflow.png"/></center>

In [2]:
# Importing essential libraries for data analysis and visualization
import numpy as np 
import pandas as pd 
import matplotlib as plt 

## 1- Get the data
<center><img src="../workflow/pycaret-workflow_01.png"/></center>

In [12]:
# load dataset
data = pd.read_csv("../data/Heart_Attack.csv")
data.head()

,Age,Gender,Heart rate,Systolic blood pressure,Diastolic blood pressure,Blood sugar,CK-MB,Troponin,Outcome
0,64,Male,66.0,160.0,83.0,160.0,1.80,0.012,Negative
1,21,Male,94.0,98.0,46.0,296.0,6.75,1.060,Positive
2,55,Male,64.0,160.0,77.0,270.0,1.99,0.003,Negative
3,64,Male,NaN,120.0,55.0,270.0,13.87,0.122,Positive
4,55,Male,64.0,112.0,65.0,300.0,1.08,0.003,Negative


## 2- Setting up the PyCaret environment

This function initializes the experiment in PyCaret and creates the transformation pipeline based on all the parameters passed in the function. The `setup` function must be called before executing any other function. It takes two required parameters: data and `target`. All the other parameters are optional. 


<center><img src="../workflow/pycaret-workflow_02.png"/></center>

--- 

### Required Parameters
There are only two required parameters in the `setup`:

- `target: float, int, str or sequence, default = -1`
If int or str, respectively index or name of the target column in data. The default value selects the last column in the dataset. If sequence, it should have shape (n_samples,). 

- `data_func: Callable[[], DATAFRAME_LIKE] = None`

The function that generate `data` (the dataframe-like input). This is useful when the dataset is large, and you need parallel operations such as `compare_models`. It can avoid broadcasting large dataset from driver to workers. Notice one and only one of `data` and `data_func must` be set.


- `data: dataframe-like = None`
Data set with shape (n_samples, n_features), where n_samples is the number of samples and n_features is the number of features. If data is not a pandas dataframe, it's converted to one using default column names.



In [14]:
# init setup
from pycaret.classification import *
clf = setup(data = data, target = 'Outcome', session_id = 123)

,Description,Value
0,Session id,123
1,Target,Outcome
2,Target type,Binary
3,Target mapping,"Negative: 0, Positive: 1"
4,Original data shape,"(1313, 9)"
5,Transformed data shape,"(1313, 9)"
6,Transformed train set shape,"(919, 9)"
7,Transformed test set shape,"(394, 9)"
8,Numeric features,7
9,Categorical features,1


### 🔍 PyCaret Setup Summary

This summary outlines the configuration and automatic preprocessing steps performed by the `setup()` function in the PyCaret classification module.

| **Description**               | **Value**                        | **Explanation** |
|------------------------------|----------------------------------|-----------------|
| **Session id**               | `123`                            | A random seed set for reproducibility. Ensures consistent results across runs. |
| **Target**                   | `Outcome`                        | The dependent variable to be predicted (e.g., disease presence). |
| **Target type**              | `Binary`                         | Indicates a binary classification problem (e.g., 0 = Negative, 1 = Positive). |
| **Target mapping**           | `Negative: 0, Positive: 1`       | Mapping of the class labels to numeric values for model compatibility. |
| **Original data shape**      | `(1313, 9)`                      | The dataset contains 1313 rows and 9 columns prior to transformation. |
| **Transformed data shape**   | `(1313, 9)`                      | Shape of the dataset after preprocessing (no rows were dropped). |
| **Transformed train set**    | `(919, 9)`                       | Size of the training set after splitting. |
| **Transformed test set**     | `(394, 9)`                       | Size of the test set (default split is 70/30). |
| **Numeric features**         | `7`                              | Number of continuous/numeric variables. |
| **Categorical features**     | `1`                              | Number of categorical (nominal/ordinal) features. |
| **Rows with missing values** | `1.4%`                           | Proportion of rows in the dataset containing missing values. |
| **Preprocess**               | `True`                           | Indicates whether PyCaret will perform automatic preprocessing. |
| **Imputation type**          | `simple`                         | A basic imputation strategy is used to handle missing values. |
| **Numeric imputation**       | `mean`                           | Missing values in numeric columns are filled using the mean of the column. |
| **Categorical imputation**   | `mode`                           | Missing values in categorical columns are filled using the most frequent value. |
| **Maximum one-hot encoding** | `25`                             | Categorical features with ≤25 unique values will be one-hot encoded. |
| **Encoding method**          | `None`                           | No advanced encoding method (e.g., target or ordinal) was applied. |
| **Fold Generator**           | `StratifiedKFold`                | Cross-validation method used to preserve the class distribution in each fold. |
| **Fold Number**              | `10`                             | The number of folds used in cross-validation. |
| **CPU Jobs**                 | `-1`                             | Utilizes all available CPU cores for parallel processing. |
| **Use GPU**                  | `False`                          | GPU acceleration is not enabled for model training. |
| **Log Experiment**           | `False`                          | Experiment tracking/logging is not activated. |
| **Experiment Name**          | `clf-default-name`               | Default name assigned to the current classification experiment. |
| **USI**                      | `343d`                           | Unique session identifier for reproducibility and reference. |

---

### 🧠 Interpretation

The dataset was successfully prepared for binary classification using PyCaret. Key steps included:

- **Simple imputation** for handling 1.4% missing values.
- **Stratified cross-validation** for balanced evaluation.
- **No complex encoding** methods were needed due to limited categorical features.

This setup provides a strong foundation for training and comparing multiple machine learning models with minimal manual preprocessing.


In [16]:
# To retrieve all current config settings
get_config()

{'USI',
 'X',
 'X_test',
 'X_test_transformed',
 'X_train',
 'X_train_transformed',
 'X_transformed',
 '_available_plots',
 '_ml_usecase',
 'data',
 'dataset',
 'dataset_transformed',
 'exp_id',
 'exp_name_log',
 'fix_imbalance',
 'fold_generator',
 'fold_groups_param',
 'fold_shuffle_param',
 'gpu_n_jobs_param',
 'gpu_param',
 'html_param',
 'idx',
 'is_multiclass',
 'log_plots_param',
 'logging_param',
 'memory',
 'n_jobs_param',
 'pipeline',
 'seed',
 'target_param',
 'test',
 'test_transformed',
 'train',
 'train_transformed',
 'variable_and_property_keys',
 'variables',
 'y',
 'y_test',
 'y_test_transformed',
 'y_train',
 'y_train_transformed',
 'y_transformed'}

In [23]:
# check the data 
data.head()

,Age,Gender,Heart rate,Systolic blood pressure,Diastolic blood pressure,Blood sugar,CK-MB,Troponin,Outcome
0,64,Male,66.0,160.0,83.0,160.0,1.80,0.012,Negative
1,21,Male,94.0,98.0,46.0,296.0,6.75,1.060,Positive
2,55,Male,64.0,160.0,77.0,270.0,1.99,0.003,Negative
3,64,Male,NaN,120.0,55.0,270.0,13.87,0.122,Positive
4,55,Male,64.0,112.0,65.0,300.0,1.08,0.003,Negative


In [21]:
# Get features/independent variables 
get_config("X")

,Age,Gender,Heart rate,Systolic blood pressure,Diastolic blood pressure,Blood sugar,CK-MB,Troponin
618,54,Male,89.0,126.0,64.0,153.0,8.32,0.009
293,65,Male,72.0,100.0,57.0,93.0,3.46,0.106
1237,23,Male,96.0,147.0,84.0,82.0,12.57,0.005
641,70,Female,75.0,150.0,95.0,91.0,2.51,0.015
212,53,Female,80.0,150.0,52.0,188.0,3.24,0.007
...,...,...,...,...,...,...,...,...
321,86,Male,81.0,125.0,79.0,114.0,2.24,0.018
73,65,Female,61.0,124.0,62.0,141.0,1.53,0.105
200,63,Female,75.0,116.0,73.0,95.0,4.69,0.006
392,39,Female,86.0,158.0,104.0,97.0,5.36,0.003


In [22]:
# Get target/dependent variable 
get_config('y')

618     Positive
293     Positive
1237    Positive
641     Positive
212     Negative
          ...   
321     Positive
73      Positive
200     Negative
392     Positive
54      Positive
Name: Outcome, Length: 1313, dtype: category
Categories (2, object): ['Negative', 'Positive']

In [17]:
# Get training features
get_config('X_train')

,Age,Gender,Heart rate,Systolic blood pressure,Diastolic blood pressure,Blood sugar,CK-MB,Troponin
618,54,Male,89.0,126.0,64.0,153.0,8.320000,0.009
293,65,Male,72.0,100.0,57.0,93.0,3.460000,0.106
1237,23,Male,96.0,147.0,84.0,82.0,12.570000,0.005
641,70,Female,75.0,150.0,95.0,91.0,2.510000,0.015
212,53,Female,80.0,150.0,52.0,188.0,3.240000,0.007
...,...,...,...,...,...,...,...,...
297,41,Male,92.0,147.0,78.0,155.0,7.010000,0.003
79,58,Female,72.0,138.0,86.0,189.0,12.020000,1.070
465,75,Male,60.0,113.0,52.0,98.0,1.610000,0.005
1310,45,Male,85.0,168.0,104.0,96.0,1.240000,4.250


In [18]:
# Get test features
get_config('X_test')

,Age,Gender,Heart rate,Systolic blood pressure,Diastolic blood pressure,Blood sugar,CK-MB,Troponin
1075,35,Male,66.0,94.0,63.0,109.0,3.71,0.003
489,41,Female,82.0,107.0,57.0,135.0,1.03,0.003
348,63,Female,51.0,130.0,70.0,77.0,3.20,0.006
237,65,Female,62.0,101.0,64.0,89.0,2.27,0.598
176,61,Male,60.0,138.0,81.0,321.0,6.17,0.001
...,...,...,...,...,...,...,...,...
321,86,Male,81.0,125.0,79.0,114.0,2.24,0.018
73,65,Female,61.0,124.0,62.0,141.0,1.53,0.105
200,63,Female,75.0,116.0,73.0,95.0,4.69,0.006
392,39,Female,86.0,158.0,104.0,97.0,5.36,0.003


In [19]:
# Get target training labels
get_config('y_train')

618     Positive
293     Positive
1237    Positive
641     Positive
212     Negative
          ...   
297     Positive
79      Positive
465     Negative
1310    Positive
53      Positive
Name: Outcome, Length: 919, dtype: category
Categories (2, object): ['Negative', 'Positive']

In [20]:
# Get full pipeline
get_config('pipeline')

Pipeline(memory=FastMemory(location=C:\Users\JHossain\AppData\Local\Temp\joblib),
         steps=[('label_encoding',
                 TransformerWrapperWithInverse(exclude=None, include=None,
                                               transformer=LabelEncoder())),
                ('numerical_imputer',
                 TransformerWrapper(exclude=None,
                                    include=['Age', 'Heart rate',
                                             'Systolic blood pressure',
                                             'Diastolic blood pressure',
                                             'Blood sugar', 'CK-MB',
                                             'Troponin...
                                    transformer=OrdinalEncoder(cols=['Gender'],
                                                               drop_invariant=False,
                                                               handle_missing='return_nan',
                                                               handle_unknown='value',
                                                               mapping=[{'col': 'Gender',
                                                                         'data_type': dtype('O'),
                                                                         'mapping': Female    0
Male      1
NaN      -1
dtype: int64}],
                                                               return_df=True,
                                                               verbose=0))),
                ('clean_column_names',
                 TransformerWrapper(exclude=None, include=None,
                                    transformer=CleanColumnNames(match='[\\]\\[\\,\\{\\}\\"\\:]+')))],
         verbose=False)

## 3- Compare Models
<center><img src="../workflow/pycaret-workflow_03.png"/></center>

In [15]:
# check available models 
models()

,Name,Reference,Turbo
ID,,,
lr,Logistic Regression,sklearn.linear_model._logistic.LogisticRegression,True
knn,K Neighbors Classifier,sklearn.neighbors._classification.KNeighborsCl...,True
nb,Naive Bayes,sklearn.naive_bayes.GaussianNB,True
dt,Decision Tree Classifier,sklearn.tree._classes.DecisionTreeClassifier,True
svm,SVM - Linear Kernel,sklearn.linear_model._stochastic_gradient.SGDC...,True
rbfsvm,SVM - Radial Kernel,sklearn.svm._classes.SVC,False
gpc,Gaussian Process Classifier,sklearn.gaussian_process._gpc.GaussianProcessC...,False
mlp,MLP Classifier,sklearn.neural_network._multilayer_perceptron....,False
ridge,Ridge Classifier,sklearn.linear_model._ridge.RidgeClassifier,True


In [16]:
# check evaluation metrics 
get_metrics()

,Name,Display Name,Score Function,Scorer,Target,Args,Greater is Better,Multiclass,Custom
ID,,,,,,,,,
acc,Accuracy,Accuracy,<function accuracy_score at 0x000001E2F0F609A0>,accuracy,pred,{},True,True,False
auc,AUC,AUC,<pycaret.internal.metrics.BinaryMulticlassScor...,"make_scorer(roc_auc_score, response_method=('d...",pred_proba,"{'average': 'weighted', 'multi_class': 'ovr'}",True,True,False
recall,Recall,Recall,<pycaret.internal.metrics.BinaryMulticlassScor...,"make_scorer(recall_score, response_method='pre...",pred,{'average': 'weighted'},True,True,False
precision,Precision,Prec.,<pycaret.internal.metrics.BinaryMulticlassScor...,"make_scorer(precision_score, response_method='...",pred,{'average': 'weighted'},True,True,False
f1,F1,F1,<pycaret.internal.metrics.BinaryMulticlassScor...,"make_scorer(f1_score, response_method='predict...",pred,{'average': 'weighted'},True,True,False
kappa,Kappa,Kappa,<function cohen_kappa_score at 0x000001E2F0F60...,"make_scorer(cohen_kappa_score, response_method...",pred,{},True,True,False
mcc,MCC,MCC,<function matthews_corrcoef at 0x000001E2F0F60...,"make_scorer(matthews_corrcoef, response_method...",pred,{},True,True,False


In [3]:
# compare models
best = compare_models()

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lr,Logistic Regression,0.7689,0.8047,0.5602,0.7208,0.6279,0.4641,0.4736,0.6310
ridge,Ridge Classifier,0.7670,0.8060,0.5497,0.7235,0.6221,0.4581,0.4690,0.0100
lda,Linear Discriminant Analysis,0.7670,0.8055,0.5550,0.7202,0.6243,0.4594,0.4695,0.0050
rf,Random Forest Classifier,0.7485,0.7911,0.5284,0.6811,0.5924,0.4150,0.4238,0.0290
nb,Naive Bayes,0.7427,0.7955,0.5702,0.6543,0.6043,0.4156,0.4215,0.0050
catboost,CatBoost Classifier,0.7410,0.7993,0.5278,0.6630,0.5851,0.4005,0.4078,0.5020
gbc,Gradient Boosting Classifier,0.7373,0.7914,0.5550,0.6445,0.5931,0.4013,0.4059,0.0210
ada,Ada Boost Classifier,0.7372,0.7799,0.5275,0.6585,0.5796,0.3926,0.4017,0.0160
et,Extra Trees Classifier,0.7299,0.7788,0.4965,0.6516,0.5596,0.3706,0.3802,0.0250
qda,Quadratic Discriminant Analysis,0.7282,0.7894,0.5281,0.6558,0.5736,0.3785,0.3910,0.0060


Processing:   0%|          | 0/65 [00:00<?, ?it/s]

In [5]:
# Change the sort order
best = compare_models(sort = 'F1')

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lr,Logistic Regression,0.7689,0.8047,0.5602,0.7208,0.6279,0.4641,0.4736,0.0070
lda,Linear Discriminant Analysis,0.7670,0.8055,0.5550,0.7202,0.6243,0.4594,0.4695,0.0050
ridge,Ridge Classifier,0.7670,0.8060,0.5497,0.7235,0.6221,0.4581,0.4690,0.0050
nb,Naive Bayes,0.7427,0.7955,0.5702,0.6543,0.6043,0.4156,0.4215,0.0050
gbc,Gradient Boosting Classifier,0.7373,0.7914,0.5550,0.6445,0.5931,0.4013,0.4059,0.0210
rf,Random Forest Classifier,0.7485,0.7911,0.5284,0.6811,0.5924,0.4150,0.4238,0.0280
catboost,CatBoost Classifier,0.7410,0.7993,0.5278,0.6630,0.5851,0.4005,0.4078,0.5240
ada,Ada Boost Classifier,0.7372,0.7799,0.5275,0.6585,0.5796,0.3926,0.4017,0.0160
qda,Quadratic Discriminant Analysis,0.7282,0.7894,0.5281,0.6558,0.5736,0.3785,0.3910,0.0050
lightgbm,Light Gradient Boosting Machine,0.7133,0.7645,0.5398,0.6036,0.5650,0.3534,0.3580,0.0380


Processing:   0%|          | 0/65 [00:00<?, ?it/s]

Notice that the sort order of scoring grid is changed now and also the best model returned by this function is selected based on F1.

In [6]:
print(best)

LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
                   intercept_scaling=1, l1_ratio=None, max_iter=1000,
                   multi_class='auto', n_jobs=None, penalty='l2',
                   random_state=123, solver='lbfgs', tol=0.0001, verbose=0,
                   warm_start=False)


In [7]:
# Compare only a few models
best = compare_models(include = ['lr', 'dt', 'lightgbm'])

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lr,Logistic Regression,0.7689,0.8047,0.5602,0.7208,0.6279,0.4641,0.4736,0.0070
lightgbm,Light Gradient Boosting Machine,0.7133,0.7645,0.5398,0.6036,0.5650,0.3534,0.3580,0.0350
dt,Decision Tree Classifier,0.6928,0.6512,0.5137,0.5636,0.5328,0.3070,0.3098,0.0060


Processing:   0%|          | 0/17 [00:00<?, ?it/s]

Alternatively, you can also use exclude parameter. This will compare all models except for the ones that are passed in the `exclude` parameter.

In [8]:
# exclude models 
best = compare_models(exclude = ['lr', 'dt', 'lightgbm'])

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
ridge,Ridge Classifier,0.7670,0.8060,0.5497,0.7235,0.6221,0.4581,0.4690,0.0050
lda,Linear Discriminant Analysis,0.7670,0.8055,0.5550,0.7202,0.6243,0.4594,0.4695,0.0050
rf,Random Forest Classifier,0.7485,0.7911,0.5284,0.6811,0.5924,0.4150,0.4238,0.0290
nb,Naive Bayes,0.7427,0.7955,0.5702,0.6543,0.6043,0.4156,0.4215,0.0050
catboost,CatBoost Classifier,0.7410,0.7993,0.5278,0.6630,0.5851,0.4005,0.4078,0.5190
gbc,Gradient Boosting Classifier,0.7373,0.7914,0.5550,0.6445,0.5931,0.4013,0.4059,0.0210
ada,Ada Boost Classifier,0.7372,0.7799,0.5275,0.6585,0.5796,0.3926,0.4017,0.0160
et,Extra Trees Classifier,0.7299,0.7788,0.4965,0.6516,0.5596,0.3706,0.3802,0.0250
qda,Quadratic Discriminant Analysis,0.7282,0.7894,0.5281,0.6558,0.5736,0.3785,0.3910,0.0060
knn,K Neighbors Classifier,0.7001,0.7164,0.5020,0.5982,0.5413,0.3209,0.3271,0.0090


Processing:   0%|          | 0/53 [00:00<?, ?it/s]

In [9]:
# Return more than one model
best = compare_models(n_select = 3)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lr,Logistic Regression,0.7689,0.8047,0.5602,0.7208,0.6279,0.4641,0.4736,0.0080
ridge,Ridge Classifier,0.7670,0.8060,0.5497,0.7235,0.6221,0.4581,0.4690,0.0060
lda,Linear Discriminant Analysis,0.7670,0.8055,0.5550,0.7202,0.6243,0.4594,0.4695,0.0050
rf,Random Forest Classifier,0.7485,0.7911,0.5284,0.6811,0.5924,0.4150,0.4238,0.0310
nb,Naive Bayes,0.7427,0.7955,0.5702,0.6543,0.6043,0.4156,0.4215,0.0050
catboost,CatBoost Classifier,0.7410,0.7993,0.5278,0.6630,0.5851,0.4005,0.4078,0.5290
gbc,Gradient Boosting Classifier,0.7373,0.7914,0.5550,0.6445,0.5931,0.4013,0.4059,0.0200
ada,Ada Boost Classifier,0.7372,0.7799,0.5275,0.6585,0.5796,0.3926,0.4017,0.0150
et,Extra Trees Classifier,0.7299,0.7788,0.4965,0.6516,0.5596,0.3706,0.3802,0.0270
qda,Quadratic Discriminant Analysis,0.7282,0.7894,0.5281,0.6558,0.5736,0.3785,0.3910,0.0060


Processing:   0%|          | 0/67 [00:00<?, ?it/s]

Notice that there is no change in the results, however, if you check the variable best , it will now contain a list of the top 3 models instead of just one model as seen previously. 

In [10]:
print(best)

[LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
                   intercept_scaling=1, l1_ratio=None, max_iter=1000,
                   multi_class='auto', n_jobs=None, penalty='l2',
                   random_state=123, solver='lbfgs', tol=0.0001, verbose=0,
                   warm_start=False), RidgeClassifier(alpha=1.0, class_weight=None, copy_X=True, fit_intercept=True,
                max_iter=None, positive=False, random_state=123, solver='auto',
                tol=0.0001), LinearDiscriminantAnalysis(covariance_estimator=None, n_components=None,
                           priors=None, shrinkage=None, solver='svd',
                           store_covariance=False, tol=0.0001)]


In [11]:
# Set the budget time
best = compare_models(budget_time = 0.5)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lr,Logistic Regression,0.7689,0.8047,0.5602,0.7208,0.6279,0.4641,0.4736,0.0080
ridge,Ridge Classifier,0.7670,0.8060,0.5497,0.7235,0.6221,0.4581,0.4690,0.0050
lda,Linear Discriminant Analysis,0.7670,0.8055,0.5550,0.7202,0.6243,0.4594,0.4695,0.0050
rf,Random Forest Classifier,0.7485,0.7911,0.5284,0.6811,0.5924,0.4150,0.4238,0.0340
nb,Naive Bayes,0.7427,0.7955,0.5702,0.6543,0.6043,0.4156,0.4215,0.0050
catboost,CatBoost Classifier,0.7410,0.7993,0.5278,0.6630,0.5851,0.4005,0.4078,0.5480
gbc,Gradient Boosting Classifier,0.7373,0.7914,0.5550,0.6445,0.5931,0.4013,0.4059,0.0210
ada,Ada Boost Classifier,0.7372,0.7799,0.5275,0.6585,0.5796,0.3926,0.4017,0.0170
et,Extra Trees Classifier,0.7299,0.7788,0.4965,0.6516,0.5596,0.3706,0.3802,0.0250
qda,Quadratic Discriminant Analysis,0.7282,0.7894,0.5281,0.6558,0.5736,0.3785,0.3910,0.0060


Processing:   0%|          | 0/65 [00:00<?, ?it/s]

If you are running short on time and would like to set a fixed budget time for this function to run, you can do that by setting the budget_time parameter.

### Set the probability threshold
When performing binary classification, you can change the probability threshold or cut-off value for hard labels. By default, all classifiers use `0.5` as a default threshold. 

In [12]:
# compare models with probability_threshold
best = compare_models(probability_threshold = 0.25)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
ridge,Ridge Classifier,0.7670,0.8060,0.5497,0.7235,0.6221,0.4581,0.4690,0.0050
nb,Naive Bayes,0.7169,0.7955,0.7096,0.5818,0.6356,0.4089,0.4185,0.0050
qda,Quadratic Discriminant Analysis,0.7038,0.7894,0.6827,0.5636,0.6155,0.3785,0.3851,0.0050
catboost,CatBoost Classifier,0.6983,0.7993,0.7901,0.5495,0.6453,0.3984,0.4225,0.5120
lda,Linear Discriminant Analysis,0.6964,0.8055,0.8012,0.5455,0.6480,0.3983,0.4224,0.0080
dt,Decision Tree Classifier,0.6928,0.6512,0.5137,0.5636,0.5328,0.3070,0.3098,0.0060
lr,Logistic Regression,0.6872,0.8047,0.8231,0.5356,0.6480,0.3896,0.4193,0.0080
gbc,Gradient Boosting Classifier,0.6870,0.7914,0.7696,0.5428,0.6342,0.3778,0.3965,0.0200
lightgbm,Light Gradient Boosting Machine,0.6833,0.7645,0.6626,0.5383,0.5924,0.3386,0.3451,0.0440
et,Extra Trees Classifier,0.6500,0.7788,0.8439,0.5010,0.6271,0.3369,0.3810,0.0310


Processing:   0%|          | 0/65 [00:00<?, ?it/s]

Notice that all metrics except for AUC are now different. AUC doesn't change because it's not dependent on the hard labels, everything else is dependent on the hard label which is now obtained using `probability_threshold=0.25`.

### Disable cross-validation
If you don't want to evaluate models using cross-validation and rather just train them and see the metrics on the test/hold-out set you can set the `cross_validation=False`.

In [13]:
best = compare_models(cross_validation=False)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
nb,Naive Bayes,0.7835,0.8523,0.6173,0.7246,0.6667,0.5079,0.5115,0.0100
rf,Random Forest Classifier,0.7662,0.8404,0.5926,0.6957,0.6400,0.4686,0.4719,0.1200
gbc,Gradient Boosting Classifier,0.7619,0.8427,0.6296,0.6711,0.6497,0.4696,0.4702,0.1200
catboost,CatBoost Classifier,0.7619,0.8512,0.5679,0.6970,0.6259,0.4539,0.4590,1.6200
lr,Logistic Regression,0.7576,0.8568,0.5309,0.7049,0.6056,0.4356,0.4447,0.0200
ridge,Ridge Classifier,0.7576,0.7026,0.5185,0.7119,0.6000,0.4322,0.4434,0.0100
lightgbm,Light Gradient Boosting Machine,0.7576,0.8248,0.6420,0.6582,0.6500,0.4646,0.4647,0.0500
lda,Linear Discriminant Analysis,0.7532,0.8563,0.5185,0.7000,0.5957,0.4238,0.4337,0.0100
et,Extra Trees Classifier,0.7532,0.8470,0.4938,0.7143,0.5839,0.4167,0.4311,0.0900
qda,Quadratic Discriminant Analysis,0.7446,0.8300,0.5062,0.6833,0.5816,0.4036,0.4130,0.0100


Processing:   0%|          | 0/65 [00:00<?, ?it/s]

## 4- Create the Model
<center><img src="../workflow/pycaret-workflow_04.png"/></center>

## 5- Tunning the Model
<center><img src="../workflow/pycaret-workflow_05.png"/></center>

## 6- Plotting the Model
<center><img src="../workflow/pycaret-workflow_06.png"/></center>

## 7- Evaluation the model
<center><img src="../workflow/pycaret-workflow_07.png"/></center>

## 8- Finalizing the Model

<center><img src="../workflow/pycaret-workflow_07.png"/></center>

## 9- Predicting with the model

<center><img src="../workflow/pycaret-workflow_09.png"/></center>

## 10- Save/Load Model for Production

<center><img src="../workflow/pycaret-workflow_10.png"/></center>